In [81]:
import numpy as np
import pandas as pd
import yfinance as yf
import ccxt

In [127]:
tickers = ["BTC-USD", "ETH-USD", "USDT-USD", "BNB-USD", "SOL-USD", "USDC-USD", "XRP-USD", "DOGE-USD", "TON-USD", "ADA-USD", "TRX-USD", "AVAX-USD", "SHIB-USD", "WBTC-USD", "DOT-USD", "LINK-USD", "BCH-USD", "NEAR-USD", "MATIC-USD", "LTC-USD", "ICP-USD", "UNI-USD", "DAI-USD", "APT-USD", "LEO-USD", "STETH-USD", "ETC-USD", "FET-USD", "XLM-USD", "OKB-USD", "ATOM-USD", "XMR-USD", "INJ-USD", "ARB-USD", "HBAR-USD", "IMX-USD", "FIL-USD", "CRO-USD", "MNT-USD", "OP-USD", "VET-USD", "RNDR-USD", "MKR-USD", "FDUSD-USD", "GRT-USD", "RUNE-USD", "AAVE-USD", "STX-USD", "SUI-USD", "ALGO-USD", "THETA-USD", "WIF-USD", "EGLD-USD", "FLOW-USD", "QNT-USD", "PEPE-USD", "AR-USD", "SAND-USD", "MANA-USD", "EOS-USD", "BONK-USD", "FLOKI-USD", "AXS-USD", "KAVA-USD", "CFX-USD", "GALA-USD", "FTM-USD", "XTZ-USD", "ROSE-USD", "NEO-USD", "CHZ-USD", "OSMO-USD", "ZEC-USD", "DASH-USD", "LUNC-USD", "CRV-USD", "BAT-USD", "IOTA-USD", "CAKE-USD", "ONE-USD", "ENJ-USD", "SNX-USD", "ZIL-USD", "LRC-USD", "1INCH-USD", "ANKR-USD", "HOT-USD", "SC-USD", "ICX-USD", "DYDX-USD", "OCEAN-USD", "BAND-USD", "RVN-USD", "WOO-USD", "STORJ-USD", "SKL-USD", "CELR-USD", "CELO-USD", "NMR-USD"]
res = []

In [128]:
for ticker in tickers:
    df = yf.download(ticker, period="2y")
    df["log_ret"] = np.log(df["Close"]/df["Close"].shift(1))
    df["ret_zscore"] = (df["log_ret"] - df["log_ret"].rolling(20).mean()) / df["log_ret"].rolling(20).std()
    df["future_ret"] = df["log_ret"].shift(-1)
    clean = df[["log_ret", "ret_zscore", "future_ret"]].dropna()
    corr = clean["ret_zscore"].corr(clean["future_ret"])
    if len(clean) < 500:
        continue
    clean = ticker.split("-")
    if corr <= -0.03 and "USD" not in clean[0] and clean[0] != "DAI":
        res.append(ticker)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [129]:
data = {}

In [130]:
for ticker in res:
    df = yf.download(ticker, period="2y")
    
    df["close_log_returns"] = np.log(df["Close"]/df["Close"].shift(1))
    df["sma_5"] = df["close_log_returns"].rolling(5).mean().shift(1)
    df["sma_5_dir"] = np.where(df["sma_5"] > 0, 1, np.where(df["sma_5"] < 0, -1, np.nan))
    
    clean = df[["close_log_returns", "sma_5_dir"]].dropna()
    clean["signal"] = -1 * clean["sma_5_dir"]

    c = np.log(1 - 0.001)
    position_change = clean["signal"].diff().abs()
    position_change.iloc[0] = clean["signal"].iloc[0]
    
    clean["trade_log_returns"] = clean["signal"] * clean["close_log_returns"] * 0.99
    clean["trade_cum_log_returns"] = clean["trade_log_returns"].cumsum()
    
    clean["strategy_simple_returns"] = np.exp(clean["trade_log_returns"]) - 1
    clean["equity_curve"] = np.exp(clean["trade_cum_log_returns"])
    
    ret = clean["equity_curve"].iloc[-1] - 1
    bh_ret = np.exp(clean["close_log_returns"].sum()) - 1

    days = len(clean)
    ann_ret = (1 + ret) ** (365 / days) - 1
    ann_vol = clean["strategy_simple_returns"].std() * np.sqrt(365)

    mean_daily_ret = clean["strategy_simple_returns"].mean()
    std_daily_ret = clean["strategy_simple_returns"].std()
    if std_daily_ret != 0:
        sharpe_ratio = (mean_daily_ret / std_daily_ret) * np.sqrt(252)
    else:
        sharpe_ratio = 0.0

    roll_max = clean["equity_curve"].cummax()
    drawdown = (clean["equity_curve"] - roll_max) / roll_max
    max_drawdown = drawdown.min()

    winning_days = (clean["strategy_simple_returns"] > 0).sum()
    active_days = (clean["strategy_simple_returns"] != 0).sum()
    win_rate = winning_days / active_days if active_days > 0 else 0
    
    data[ticker] = {
        "Total Return": ret, 
        "B&H Return": bh_ret,
        "Annualized Return": ann_ret,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe_ratio,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate
    }

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [132]:
results_df = pd.DataFrame(data).T

In [133]:
results_df = results_df.round(4)

In [134]:
results_df

,Total Return,B&H Return,Annualized Return,Annualized Volatility,Sharpe Ratio,Max Drawdown,Win Rate
TON-USD,1.325232e+26,-0.9970,1.416520e+13,179.4373,1.5160,-0.9940,0.5164
SHIB-USD,2.216160e+01,-0.6845,6.033600e+00,0.9931,2.0470,-0.3571,0.5940
BCH-USD,4.878500e+00,-0.2440,1.439400e+00,0.7248,1.3239,-0.4302,0.5490
LEO-USD,2.170400e+00,0.6839,7.877000e-01,0.4601,1.2482,-0.3009,0.5490
MKR-USD,5.301200e+00,-0.3893,1.526200e+00,0.8983,1.2314,-0.7803,0.5448
FLOW-USD,2.570000e-02,-0.9623,1.290000e-02,1.0986,0.4688,-0.6986,0.5048
QNT-USD,1.114000e-01,-0.2094,5.460000e-02,0.8305,0.3909,-0.6171,0.5021
BONK-USD,6.858000e-01,-0.8453,3.457000e-01,1.3491,0.7479,-0.8274,0.5215
XTZ-USD,1.942200e+00,-0.6573,7.217000e-01,0.8441,0.8959,-0.6836,0.5269
1INCH-USD,1.201700e+00,-0.7859,4.879000e-01,0.9116,0.7368,-0.6083,0.5131
